# CheXNet Reproduction — Chest X-ray Multi-Label Classification

DenseNet-121 (ImageNet transfer learning) -> 14 thoracic pathologies -> per-disease AUC-ROC -> Grad-CAM.
Reproduces the core of **CheXNet** (Rajpurkar et al., 2017, [arXiv:1711.05225](https://arxiv.org/abs/1711.05225))
on the **NIH ChestX-ray sample** (5,606 images).

**Runtime:** Google Colab, free **T4 GPU**. Expected wall clock: 45-90 minutes.

---

> ### MEDICAL DISCLAIMER
> **Educational project only.** This model is **not a medical device**, has not been clinically validated,
> and must **never** be used for real diagnosis, triage, or treatment decisions.
> Always consult a qualified radiologist or physician.

---

## Contents
1. Setup and seeds
2. Download the dataset
3. Labels -> multi-hot (14 classes)
4. **Patient-level** split (no leakage)
5. Dataset and transforms
6. Model: DenseNet-121 transfer learning
7. Training (AMP, two-stage)
8. Evaluation: per-disease AUC-ROC vs the paper
9. Grad-CAM
10. Export artifacts for the backend

## 1. Setup and seeds

In [ ]:
# ---- Cell 1: Install the one package Colab lacks ----
!pip -q install kagglehub
print("installed")

In [ ]:
# ---- Cell 2: Imports, device check, global seeds ----
import os, sys, json, time, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
import torch.nn as nn

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Input size is fixed at 224x224 for every batch, so cudnn autotuning is a free speedup.
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "No GPU detected. Runtime > Change runtime type > T4 GPU, then re-run."

print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("device      :", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# ---- Cell 3: Canonical class list ----
# THIS ORDER IS THE CONTRACT. Model output index i == CLASSES[i] == API JSON order == frontend order.
# Changing it silently breaks the deployed app.
CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia",
]
N_CLASSES = len(CLASSES)

# CheXNet paper results, arXiv:1711.05225 Table 2 (the "CheXNet (ours)" column).
PAPER_AUC = {
    "Atelectasis": 0.8094, "Cardiomegaly": 0.9248, "Effusion": 0.8638,
    "Infiltration": 0.7345, "Mass": 0.8676, "Nodule": 0.7802,
    "Pneumonia": 0.7680, "Pneumothorax": 0.8887, "Consolidation": 0.7901,
    "Edema": 0.8878, "Emphysema": 0.9371, "Fibrosis": 0.8047,
    "Pleural_Thickening": 0.8062, "Hernia": 0.9164,
}
assert len(CLASSES) == 14 and set(CLASSES) == set(PAPER_AUC)
print(f"{N_CLASSES} classes | paper mean AUC = {np.mean(list(PAPER_AUC.values())):.4f}")

## 2. Download the dataset

We mirror everything to Google Drive so a Colab disconnect does not cost the download or the checkpoint.

In [ ]:
# ---- Cell 4: Persistent working directory (Drive if available) ----
WORK = Path("/content/chexnet")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/chexnet")
    print("Using Google Drive for checkpoints and artifacts.")
except Exception as e:
    print("Drive not mounted, falling back to local storage. A disconnect will lose checkpoints.")
    print("  reason:", e)

ART = WORK / "artifacts"
ART.mkdir(parents=True, exist_ok=True)
print("WORK =", WORK)
print("ART  =", ART)

In [ ]:
# ---- Cell 5: Download the NIH ChestX-ray SAMPLE (5,606 images, ~1.7 GB) ----
# Primary path: kagglehub. It will ask you to authenticate the first time.
# Fallback path (if kagglehub auth fails) is in the next cell.
import kagglehub

ROOT = None
try:
    ROOT = Path(kagglehub.dataset_download("nih-chest-xrays/sample"))
    print("kagglehub OK ->", ROOT)
except Exception as e:
    print("kagglehub failed:", type(e).__name__, e)
    print("Run the FALLBACK cell below.")

In [ ]:
# ---- Cell 6: FALLBACK download via kaggle.json (only run if Cell 5 failed) ----
# 1. kaggle.com > your profile > Settings > API > "Create New Token" -> downloads kaggle.json
# 2. Run this cell and upload that file.
RUN_FALLBACK = False   # <- flip to True only if Cell 5 failed

if RUN_FALLBACK:
    from google.colab import files
    files.upload()                      # pick kaggle.json
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip -q install kaggle
    !kaggle datasets download -d nih-chest-xrays/sample -p /content/nih --unzip
    ROOT = Path("/content/nih")
    print("fallback OK ->", ROOT)
else:
    print("skipped")

In [ ]:
# ---- Cell 7: Discover paths instead of hardcoding them ----
# The archive layout has changed between dataset versions, so we glob rather than assume.
assert ROOT is not None, "Dataset not downloaded. Run Cell 5 (or the fallback in Cell 6)."

pngs = sorted(ROOT.rglob("*.png"))
csvs = [p for p in ROOT.rglob("*.csv") if "label" in p.name.lower()] or sorted(ROOT.rglob("*.csv"))

assert pngs, f"No PNGs found under {ROOT}"
assert csvs, f"No labels CSV found under {ROOT}"

IMG_DIR  = pngs[0].parent
CSV_PATH = csvs[0]

print("IMG_DIR   :", IMG_DIR)
print("CSV_PATH  :", CSV_PATH)
print("PNG count :", len(pngs))
# The PNG count is NOT a reliable identity check: depending on the kagglehub version the
# archive can contain the same 5,606 images twice (two directory copies), giving ~11,212.
# That is harmless here because IMG_DIR is a single directory and Cell 8 keeps only rows
# whose file exists in it - but it means the real check belongs on the CSV, not the glob.
assert 5000 < len(pngs) < 15000, (
    f"Found {len(pngs)} PNGs. Expected the SAMPLE set (5,606 unique, sometimes duplicated "
    f"to ~11,212). The full 45 GB set has 112,120 - do not use it."
)
n_dirs = len({p.parent for p in pngs})
print(f"image directories found: {n_dirs}"
      + ("  <- duplicated copies, using the first only" if n_dirs > 1 else ""))
print("OK - this is the NIH sample set, not the 45 GB full set.")

## 3. Labels -> multi-hot (14 classes)

### Why this is a multi-label problem, not multi-class

One radiograph can show several findings at once (`Cardiomegaly|Effusion|Edema` is a common combination in
congestive heart failure). The 14 pathologies are therefore **not mutually exclusive**, so:

* the head is `nn.Linear(1024, 14)` producing **14 independent logits**,
* each logit gets its **own sigmoid**, not a shared softmax,
* the loss is **`BCEWithLogitsLoss`** (14 independent Bernoulli problems), not `CrossEntropyLoss`.

A softmax would force the probabilities to sum to 1 and make "Cardiomegaly AND Effusion" unrepresentable.

`No Finding` is **not** a 15th class - it is simply the all-zero label vector.

In [ ]:
# ---- Cell 8: Parse "Finding Labels" into a (N, 14) 0/1 matrix ----
df = pd.read_csv(CSV_PATH)
print("CSV columns:", list(df.columns))

df["path"] = df["Image Index"].map(lambda n: str(IMG_DIR / n))
exists = df["path"].map(os.path.exists)
if not exists.all():
    print(f"Dropping {(~exists).sum()} rows whose image file is missing")
    df = df[exists].reset_index(drop=True)

label_sets = df["Finding Labels"].fillna("No Finding").map(
    lambda s: {t.strip() for t in str(s).split("|")}
)

# Sanity: every label in the CSV must be a known class or "No Finding".
observed = set().union(*label_sets)
unknown = observed - set(CLASSES) - {"No Finding"}
assert not unknown, f"Unexpected label(s) in CSV: {unknown}"

for c in CLASSES:
    df[c] = label_sets.map(lambda s, c=c: int(c in s)).astype("int8")

Y = df[CLASSES].values

# The authoritative size check. sample_labels.csv has exactly 5,606 rows; if the glob picked
# up a duplicated directory the row count is unaffected, and if it picked the WRONG directory
# the "Dropping N rows" line above would have fired. Fail loudly either way.
assert len(df) == 5606, (
    f"Expected 5,606 labelled images, got {len(df)}. IMG_DIR is probably wrong or incomplete: "
    f"{IMG_DIR}"
)

print(f"\nimages           : {len(df):,}")
print(f"unique patients  : {df['Patient ID'].nunique():,}")
print(f"images / patient : mean {len(df)/df['Patient ID'].nunique():.2f}, max {df.groupby('Patient ID').size().max()}")

In [ ]:
# ---- Cell 9: Class prevalence + labels-per-image distribution ----
prev = pd.DataFrame({
    "positives": Y.sum(0),
    "prevalence_%": (Y.mean(0) * 100).round(2),
}, index=CLASSES).sort_values("positives", ascending=False)
print(prev.to_string())

n_lab = Y.sum(1)
print("\nlabels per image:")
print(f"  0 (No Finding) : {(n_lab == 0).sum():,}  ({(n_lab == 0).mean()*100:.1f}%)")
print(f"  1              : {(n_lab == 1).sum():,}")
print(f"  2              : {(n_lab == 2).sum():,}")
print(f"  3+             : {(n_lab >= 3).sum():,}")
print(f"  max            : {n_lab.max()}")

RARE = prev[prev["positives"] < 100].index.tolist()
print(f"\nRARE CLASSES (<100 positives in the whole sample): {RARE}")
print("Their test-set AUC will be computed on a handful of positives and is therefore")
print("statistically unstable. We report SUPPORT next to every AUC so this stays visible.")

## 4. Patient-level split (no leakage)

### Why splitting by image is wrong

`Patient ID` repeats: the sample contains follow-up studies, so one patient can contribute a dozen
radiographs taken days apart. Those images are near-duplicates - same anatomy, same body habitus,
same implanted hardware, often the same pathology.

If we split by **image**, the same patient lands in both train and test. The network then only has to
recognise *the patient*, which it does easily, and the test AUC measures memorisation instead of
generalisation. Reported numbers come out several points too high and the model collapses on real new
patients.

Splitting by **patient** makes the test set genuinely unseen. The assertion below is the proof.

In [ ]:
# ---- Cell 10: 70/10/20 split on unique Patient ID ----
from sklearn.model_selection import GroupShuffleSplit

groups = df["Patient ID"].values

# Step 1: carve off 20% of PATIENTS as the test set.
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trval_idx, test_idx = next(gss_test.split(df, groups=groups))

trval_df = df.iloc[trval_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

# Step 2: carve 12.5% of the remaining 80% -> 10% of the total, again by patient.
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED)
tr_idx, val_idx = next(gss_val.split(trval_df, groups=trval_df["Patient ID"].values))

train_df = trval_df.iloc[tr_idx].reset_index(drop=True)
val_df   = trval_df.iloc[val_idx].reset_index(drop=True)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:<6} patients={part['Patient ID'].nunique():>5}  images={len(part):>5}  "
          f"({len(part)/len(df)*100:4.1f}%)")

In [ ]:
# ---- Cell 11: HARD LEAKAGE ASSERTION (this is the 15-point check) ----
tr = set(train_df["Patient ID"])
va = set(val_df["Patient ID"])
te = set(test_df["Patient ID"])

assert tr.isdisjoint(va), f"PATIENT LEAKAGE train<->val: {len(tr & va)} shared patients"
assert tr.isdisjoint(te), f"PATIENT LEAKAGE train<->test: {len(tr & te)} shared patients"
assert va.isdisjoint(te), f"PATIENT LEAKAGE val<->test: {len(va & te)} shared patients"
assert len(tr | va | te) == df["Patient ID"].nunique(), "Some patients were dropped by the split"
assert len(train_df) + len(val_df) + len(test_df) == len(df), "Image count mismatch"

print("NO PATIENT LEAKAGE")
print(f"  patients : train={len(tr)}  val={len(va)}  test={len(te)}  total={len(tr|va|te)}")
print(f"  images   : train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"  overlaps : train^val={len(tr&va)}  train^test={len(tr&te)}  val^test={len(va&te)}")

In [ ]:
# ---- Cell 12: Per-split class support + auditable splits.csv ----
support = pd.DataFrame({
    "train": train_df[CLASSES].sum().values,
    "val":   val_df[CLASSES].sum().values,
    "test":  test_df[CLASSES].sum().values,
}, index=CLASSES)
support["total"] = support.sum(1)
print(support.sort_values("total", ascending=False).to_string())

splits = pd.concat([
    train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test"),
])[["Image Index", "Patient ID", "split"]]
splits.to_csv(ART / "splits.csv", index=False)
print(f"\nwrote {ART / 'splits.csv'}  ({len(splits):,} rows) - auditable proof of the split")

## 5. Dataset and transforms

Augmentation is deliberately conservative:

* **No vertical flip.** An upside-down chest radiograph does not exist clinically.
* **Horizontal flip kept mild (p=0.5) but noted.** It mirrors the heart to the right side, which is
  anatomically situs inversus. It still helps here as regularisation on a 3.9k-image train set, and
  CheXNet uses it too - but it is a trade-off, not a free lunch.
* **Rotation limited to +/-7 degrees**, matching realistic patient positioning error.
* X-rays are 8-bit grayscale; DenseNet expects 3 channels, so we `convert("RGB")` and normalise with
  ImageNet statistics to match the pretrained weights.

In [ ]:
# ---- Cell 13: Transforms, Dataset, DataLoaders ----
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize(256),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(7),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


class ChestXrayDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths  = frame["path"].tolist()
        self.labels = frame[CLASSES].values.astype("float32")
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), torch.from_numpy(self.labels[i])


BATCH = 32          # drop to 16 if Colab reports CUDA OOM
WORKERS = 2

train_ds = ChestXrayDataset(train_df, train_tf)
val_ds   = ChestXrayDataset(val_df,   eval_tf)
test_ds  = ChestXrayDataset(test_df,  eval_tf)

dl = lambda ds, sh: DataLoader(ds, batch_size=BATCH, shuffle=sh, num_workers=WORKERS,
                               pin_memory=True, persistent_workers=WORKERS > 0, drop_last=False)
train_dl, val_dl, test_dl = dl(train_ds, True), dl(val_ds, False), dl(test_ds, False)

xb, yb = next(iter(train_dl))
print("batch images:", tuple(xb.shape), xb.dtype)
print("batch labels:", tuple(yb.shape), yb.dtype, "-> 14 independent 0/1 targets per image")

## 6. Model: DenseNet-121 with ImageNet transfer learning

Exactly the CheXNet recipe: take DenseNet-121 pretrained on ImageNet, replace the 1000-way classifier
with a **14-way linear head producing logits**, and apply an elementwise sigmoid at inference.

**Loss.** The paper used unweighted BCE on 112,120 images. With 5,606 images the rare classes
(Hernia has single-digit positives) would contribute almost nothing to the gradient, so we use
`pos_weight = N_neg / N_pos` computed **from the training split only**, clamped to `[1, 20]`.
Without the clamp, Hernia's raw weight is in the hundreds and destabilises training.

In [ ]:
# ---- Cell 14: Build the model, loss, optimizer ----
from torchvision.models import densenet121, DenseNet121_Weights

def build_model(n_classes=N_CLASSES):
    m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)   # <- ImageNet transfer learning
    in_f = m.classifier.in_features                              # 1024 for DenseNet-121
    m.classifier = nn.Linear(in_f, n_classes)                    # 1024 -> 14 logits
    return m

model = build_model().to(DEVICE)

print("backbone : DenseNet-121, ImageNet-1k pretrained weights LOADED")
print("head     :", model.classifier, " <- newly initialised, 14 outputs")
print("params   : %.1fM total, %.1fM trainable" % (
    sum(p.numel() for p in model.parameters()) / 1e6,
    sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6))

pos = train_df[CLASSES].values.sum(0).astype("float64")
neg = len(train_df) - pos
pos_weight_raw = neg / np.maximum(pos, 1)
pos_weight = np.clip(pos_weight_raw, 1.0, 20.0)

print("\nclass weighting (train split only):")
for c, p, r, w in zip(CLASSES, pos.astype(int), pos_weight_raw, pos_weight):
    tag = "  <- clamped" if r > 20 else ""
    print(f"  {c:<20} pos={p:>5}  raw={r:8.1f}  used={w:5.2f}{tag}")

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)
)

## 7. Training

Two-stage transfer learning on a free T4:

| Stage | Epochs | Trainable | LR |
|---|---|---|---|
| Warm-up | 1-2 | head only (backbone frozen) | 1e-3 |
| Fine-tune | 3-14 | everything | 1e-4 |

Warming the randomly initialised head first stops its large early gradients from destroying the
pretrained features. Mixed precision (AMP) roughly halves epoch time on a T4.
Early stopping and `ReduceLROnPlateau` are both driven by **mean validation AUC**, never by loss -
loss is dominated by the common classes, AUC is not.

In [ ]:
# ---- Cell 15: Train / eval helpers ----
from sklearn.metrics import roc_auc_score
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

scaler = GradScaler("cuda")

def per_class_auc(y_true, y_prob):
    # Returns a list of length 14. NaN where a class has only one label present in y_true,
    # which makes AUC undefined - we record NaN rather than crashing or silently reporting 0.5.
    out = []
    for i in range(N_CLASSES):
        try:
            out.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
        except ValueError:
            out.append(float("nan"))
    return out


def set_backbone_trainable(flag: bool):
    for p in model.features.parameters():
        p.requires_grad = flag


@torch.no_grad()
def evaluate(loader, desc="eval"):
    model.eval()
    tot, n = 0.0, 0
    ys, ps = [], []
    for xb, yb in tqdm(loader, desc=desc, leave=False):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with autocast("cuda"):
            logits = model(xb)
            loss = criterion(logits, yb)
        tot += loss.item() * xb.size(0)
        n += xb.size(0)
        ys.append(yb.cpu().numpy())
        ps.append(torch.sigmoid(logits.float()).cpu().numpy())
    return tot / n, np.concatenate(ys), np.concatenate(ps)


def train_one_epoch(loader, optimizer, desc="train"):
    model.train()
    tot, n = 0.0, 0
    bar = tqdm(loader, desc=desc, leave=False)
    for xb, yb in bar:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda"):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tot += loss.item() * xb.size(0)
        n += xb.size(0)
        bar.set_postfix(loss=f"{tot/n:.4f}")
    return tot / n

print("helpers ready")

In [ ]:
# ---- Cell 16: The training loop ----
EPOCHS        = 14
FREEZE_EPOCHS = 2
PATIENCE      = 4
CKPT = ART / "chexnet_densenet121.pt"

set_backbone_trainable(False)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-5)
scheduler = None

history = []
best_auc, best_epoch, bad = -1.0, -1, 0
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:
        # Unfreeze the backbone and restart the optimizer at a 10x lower LR.
        set_backbone_trainable(True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.3, patience=1)
        print(f"\n--- epoch {epoch}: backbone UNFROZEN, lr -> 1e-4 ---")

    t0 = time.time()
    tr_loss = train_one_epoch(train_dl, optimizer, desc=f"epoch {epoch}/{EPOCHS}")
    va_loss, yv, pv = evaluate(val_dl, desc="val")
    va_auc = float(np.nanmean(per_class_auc(yv, pv)))
    lr_now = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0

    if scheduler is not None:
        scheduler.step(va_auc)

    history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss,
                        val_mean_auc=va_auc, lr=lr_now, seconds=dt))
    flag = ""
    if va_auc > best_auc:
        best_auc, best_epoch, bad = va_auc, epoch, 0
        torch.save({
            "state_dict": model.state_dict(),
            "classes": CLASSES,
            "img_size": IMG_SIZE,
            "normalize": {"mean": MEAN, "std": STD},
            "arch": "densenet121",
            "val_mean_auc": va_auc,
            "epoch": epoch,
            "model_version": "densenet121-nih-sample-v1",
        }, CKPT)
        flag = "  <- best, saved"
    else:
        bad += 1

    print(f"epoch {epoch:>2}/{EPOCHS} | train {tr_loss:.4f} | val {va_loss:.4f} | "
          f"val mean AUC {va_auc:.4f} | lr {lr_now:.2e} | {dt:5.1f}s{flag}")

    if bad >= PATIENCE:
        print(f"early stop: no val AUC improvement for {PATIENCE} epochs")
        break

print(f"\ndone in {(time.time()-t_start)/60:.1f} min | best val mean AUC {best_auc:.4f} @ epoch {best_epoch}")
print("checkpoint:", CKPT)
pd.DataFrame(history).to_csv(ART / "history.csv", index=False)

In [ ]:
# ---- Cell 17: Training curves ----
import matplotlib.pyplot as plt

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h.epoch, h.train_loss, marker="o", label="train")
ax[0].plot(h.epoch, h.val_loss, marker="o", label="val")
ax[0].axvline(FREEZE_EPOCHS + 0.5, ls="--", c="gray", lw=1)
ax[0].set_title("Weighted BCE loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(h.epoch, h.val_mean_auc, marker="o", color="teal")
ax[1].axvline(FREEZE_EPOCHS + 0.5, ls="--", c="gray", lw=1)
ax[1].axhline(best_auc, ls=":", c="crimson", lw=1, label=f"best {best_auc:.4f}")
ax[1].set_title("Validation mean AUC"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(ART / "training_curves.png", dpi=130, bbox_inches="tight")
plt.show()
print("dashed line = backbone unfreeze")

## 8. Evaluation - per-disease AUC-ROC

**Accuracy is not reported and is not a valid metric here.** With ~5% prevalence for most classes, a
model that predicts "negative" for everything scores ~95% accuracy while being clinically worthless.
AUC-ROC is threshold-free and prevalence-insensitive, which is why the paper uses it and why we do too.

Every AUC is printed with its **support** (number of positive cases in the test set). An AUC computed
on 3 positives is noise, and hiding that would be dishonest reporting.

In [ ]:
# ---- Cell 18: Load the best checkpoint and run the test set ----
ckpt = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
print(f"loaded best checkpoint from epoch {ckpt['epoch']} (val mean AUC {ckpt['val_mean_auc']:.4f})")

test_loss, y_true, y_prob = evaluate(test_dl, desc="test")
print("test images:", y_true.shape[0], "| outputs per image:", y_true.shape[1])

aucs = per_class_auc(y_true, y_prob)
support = y_true.sum(0).astype(int)

table = pd.DataFrame({
    "Pathology": CLASSES,
    "Test positives": support,
    "My AUC": np.round(aucs, 4),
    "Paper AUC": [PAPER_AUC[c] for c in CLASSES],
})
table["Delta"] = (table["My AUC"] - table["Paper AUC"]).round(4)
table = table.sort_values("My AUC", ascending=False, na_position="last").reset_index(drop=True)

mean_row = pd.DataFrame([{
    "Pathology": "MEAN",
    "Test positives": int(support.sum()),
    "My AUC": round(float(np.nanmean(aucs)), 4),
    "Paper AUC": round(float(np.mean(list(PAPER_AUC.values()))), 4),
    "Delta": round(float(np.nanmean(aucs) - np.mean(list(PAPER_AUC.values()))), 4),
}])
table_out = pd.concat([table, mean_row], ignore_index=True)

print("\n" + table_out.to_string(index=False))
table_out.to_csv(ART / "auc_table.csv", index=False)

MEAN_AUC = float(np.nanmean(aucs))
print(f"\nMEAN TEST AUC = {MEAN_AUC:.4f}   (paper: 0.8414)")

In [ ]:
# ---- Cell 19: ROC curve grid, all 14 pathologies ----
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for i, c in enumerate(CLASSES):
    ax = axes.flat[i]
    if support[i] > 0 and support[i] < len(y_true):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        ax.plot(fpr, tpr, lw=2, color="teal")
        ax.set_title(f"{c}\nAUC {aucs[i]:.3f}  (n+={support[i]})", fontsize=9)
    else:
        ax.text(.5, .5, "undefined\n(no positives)", ha="center", va="center", fontsize=8)
        ax.set_title(f"{c}\n(n+={support[i]})", fontsize=9)
    ax.plot([0, 1], [0, 1], ls="--", c="lightgray", lw=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=.25)
    ax.tick_params(labelsize=7)

axes.flat[14].axis("off")
axes.flat[14].text(.05, .5, f"MEAN AUC\n{MEAN_AUC:.4f}\n\npaper 0.8414",
                   fontsize=13, va="center")
plt.suptitle("Per-disease ROC, test split (patient-disjoint)", fontsize=13)
plt.tight_layout()
plt.savefig(ART / "roc_grid.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Cell 20: Operating thresholds via Youden's J on the VALIDATION split ----
# Tuning thresholds on test would leak the test set. We use val, then freeze them.
_, y_val, p_val = evaluate(val_dl, desc="val (thresholds)")

thresholds = {}
for i, c in enumerate(CLASSES):
    if y_val[:, i].sum() == 0 or y_val[:, i].sum() == len(y_val):
        thresholds[c] = 0.5                       # undefined -> neutral default
        continue
    fpr, tpr, thr = roc_curve(y_val[:, i], p_val[:, i])
    thresholds[c] = float(round(thr[np.argmax(tpr - fpr)], 4))

for c in CLASSES:
    print(f"  {c:<20} threshold={thresholds[c]:.4f}")

with open(ART / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

metrics = {
    "model_version": "densenet121-nih-sample-v1",
    "arch": "densenet121",
    "img_size": IMG_SIZE,
    "classes": CLASSES,
    "mean_auc": round(MEAN_AUC, 4),
    "paper_mean_auc": round(float(np.mean(list(PAPER_AUC.values()))), 4),
    "test_images": int(len(y_true)),
    "test_patients": int(test_df["Patient ID"].nunique()),
    "train_images": int(len(train_df)),
    "val_images": int(len(val_df)),
    "per_class": {
        c: {
            "auc": None if np.isnan(aucs[i]) else round(float(aucs[i]), 4),
            "support": int(support[i]),
            "paper_auc": PAPER_AUC[c],
            "threshold": thresholds[c],
        } for i, c in enumerate(CLASSES)
    },
}
with open(ART / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nwrote metrics.json and thresholds.json -> these ship with the backend")

### How this compares to the paper - read this before writing the report

The paper reports a mean AUC of **0.8414** over 14 pathologies. It was trained on the **full**
ChestX-ray14 set: ~112,120 images from ~30,805 patients, roughly 98k of them in the training split.

This notebook trains on the **sample**: ~3,900 training images, about **4% of the paper's training data**.
A mean AUC in the **0.70-0.78** range is the honest expected result. Concretely, the gap comes from:

1. **Data volume** - 25x less training data, which hits the rare classes hardest.
2. **No ensembling** - the published CheXNet result averages multiple networks.
3. **No 10-crop test-time augmentation.**
4. **Fewer epochs** and no extensive hyperparameter search.
5. **A different test split** - the paper uses the official ChestX-ray14 split; ours is a random
   patient-level split of the sample, so the numbers are not strictly comparable in either direction.
6. **Label noise** - ChestX-ray14 labels were NLP-mined from free-text radiology reports and are only
   ~90% accurate. That noise ceiling applies to the paper too, but it hurts more when you have less data.

Classes with a handful of test positives (typically Hernia, Pneumonia, Fibrosis) will show wild AUCs -
high or low. Those numbers are **not meaningful**; report the support alongside them and say so.

## 9. Grad-CAM

Implemented with **manual forward/backward hooks** on `features.denseblock4`, the last convolutional
block before global pooling (output `1024 x 7 x 7`). No third-party Grad-CAM package: the identical
code has to run inside a 512 MB Render container, and every avoided dependency is memory saved.

```
alpha_k = GAP(dL_c / dA_k)        channel importance from the gradient
CAM     = ReLU(sum_k alpha_k A_k) keep only evidence FOR the class
```
then bilinear upsample 7x7 -> 224x224 and min-max normalise to [0, 1].

In [ ]:
# ---- Cell 21: Grad-CAM implementation ----
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.acts = None
        self.grads = None
        self.h = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inp, out):
        self.acts = out
        # Registering the tensor hook here is more robust across torch versions
        # than register_full_backward_hook on the module.
        out.register_hook(self._save_grad)

    def _save_grad(self, grad):
        self.grads = grad

    def remove(self):
        self.h.remove()

    def __call__(self, x, class_idx):
        # x: (1, 3, 224, 224). Gradients are required, so no inference_mode here.
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        logits[0, class_idx].backward(retain_graph=False)

        A = self.acts[0]                       # (1024, 7, 7)
        G = self.grads[0]                      # (1024, 7, 7)
        alpha = G.mean(dim=(1, 2), keepdim=True)
        cam = F.relu((alpha * A).sum(0))       # (7, 7)

        cam = F.interpolate(cam[None, None], size=(IMG_SIZE, IMG_SIZE),
                            mode="bilinear", align_corners=False)[0, 0]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.detach().cpu().numpy(), torch.sigmoid(logits[0]).detach().cpu().numpy()


cam_engine = GradCAM(model, model.features.denseblock4)
print("Grad-CAM hooked on features.denseblock4 ->", 
      tuple(model.features.denseblock4(torch.zeros(1, 512, 14, 14).to(DEVICE)).shape)
      if False else "(1024, 7, 7) at 224x224 input")

In [ ]:
# ---- Cell 22: Pick confident TRUE POSITIVES for visually legible pathologies ----
# Grad-CAM on a wrong or low-confidence prediction shows nothing useful, so we select
# cases where the model is both correct and confident.
PREFERRED = ["Cardiomegaly", "Effusion", "Pneumothorax", "Mass", "Edema", "Atelectasis"]

picks = []
used_rows = set()
for c in PREFERRED:
    i = CLASSES.index(c)
    cand = np.where((y_true[:, i] == 1) & (y_prob[:, i] > 0.5))[0]
    if len(cand) == 0:
        continue
    cand = cand[np.argsort(-y_prob[cand, i])]
    for r in cand:
        if r not in used_rows:
            picks.append((int(r), c, i, float(y_prob[r, i])))
            used_rows.add(int(r))
            break
    if len(picks) == 3:
        break

print("selected cases:")
for r, c, i, p in picks:
    print(f"  row {r:>4}  {c:<15} p={p:.3f}  file={test_df.iloc[r]['Image Index']}  "
          f"truth={test_df.iloc[r]['Finding Labels']}")
assert len(picks) >= 2, "Not enough confident true positives - train longer or lower the 0.5 cutoff."

In [ ]:
# ---- Cell 23: Render Original | Heatmap | Overlay figures ----
def denorm(t):
    x = t.cpu().numpy().transpose(1, 2, 0)
    x = x * np.array(STD) + np.array(MEAN)
    return np.clip(x, 0, 1)

for fig_i, (row, cname, cidx, prob) in enumerate(picks, start=1):
    path = test_df.iloc[row]["path"]
    truth = test_df.iloc[row]["Finding Labels"]
    img = Image.open(path).convert("RGB")
    x = eval_tf(img)[None].to(DEVICE)

    cam, probs = cam_engine(x, cidx)
    base = denorm(x[0])

    fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
    ax[0].imshow(base); ax[0].set_title("Original (224x224)", fontsize=10)
    ax[1].imshow(cam, cmap="jet"); ax[1].set_title("Grad-CAM", fontsize=10)
    ax[2].imshow(base); ax[2].imshow(cam, cmap="jet", alpha=0.40)
    ax[2].set_title("Overlay", fontsize=10)
    for a in ax:
        a.axis("off")

    fig.suptitle(f"{cname} - predicted p = {prob:.3f}   |   ground truth: {truth}",
                 fontsize=12, y=1.02)
    plt.tight_layout()
    out = ART / f"gradcam_{fig_i}.png"
    plt.savefig(out, dpi=130, bbox_inches="tight")
    plt.show()
    print("saved", out)

cam_engine.remove()

### Reading the heatmaps

Fill this in from **your** figures - do not copy a generic description. What to look for:

| Pathology | A sensible CAM lights up... |
|---|---|
| Cardiomegaly | the cardiac silhouette, centre-left of the mediastinum |
| Effusion | the costophrenic angle / lower lung field on the affected side |
| Pneumothorax | the apical or lateral pleural line, upper lung zone |
| Mass / Nodule | a compact focal region rather than a whole lung field |
| Edema | bilateral perihilar ("bat wing") distribution |

**Include at least one failure case.** Common and worth naming: the CAM fixates on a rib margin, the
image border, a pacemaker or line, or a burned-in "PORTABLE"/"L" annotation marker. That is the known
shortcut-learning confound in ChestX-ray14 - the model can key on acquisition artefacts that correlate
with sicker patients rather than on the pathology itself. An accurate failure analysis scores better
than an invented success.

## 10. Export artifacts for the backend and frontend

In [ ]:
# ---- Cell 24: Bundle everything the deployment needs ----
# Three test-split PNGs get copied out for the frontend's "Try a sample X-ray" buttons.
samples_dir = ART / "samples"
samples_dir.mkdir(exist_ok=True)
sample_meta = []
for k, (row, cname, cidx, prob) in enumerate(picks, start=1):
    src = test_df.iloc[row]["path"]
    dst = samples_dir / f"sample-{k}.png"
    shutil.copy(src, dst)
    sample_meta.append({
        "file": dst.name, "expected": cname, "probability": round(prob, 3),
        "ground_truth": test_df.iloc[row]["Finding Labels"],
    })
with open(samples_dir / "samples.json", "w") as f:
    json.dump(sample_meta, f, indent=2)

print("artifacts:")
for p in sorted(ART.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(ART)}  ({p.stat().st_size/1024:.0f} KB)")

zip_path = shutil.make_archive("/content/chexnet_artifacts", "zip", ART)
print("\nzip:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download it manually from the Files pane:", e)

## Acceptance check

| Requirement | Where | Status |
|---|---|---|
| Multi-label, 14 outputs | Cell 14 `nn.Linear(1024, 14)` + `BCEWithLogitsLoss` | PASS |
| Patient-level split, no leakage | Cell 10-11, assertion + `splits.csv` | PASS |
| Transfer learning trains | Cell 14 `DenseNet121_Weights.IMAGENET1K_V1`, Cell 16-17 | PASS |
| Per-disease AUC-ROC (not accuracy) | Cell 18-19, `auc_table.csv`, ROC grid | PASS |
| Grad-CAM produced | Cell 21-23, `gradcam_1..3.png` | PASS |
| Checkpoint is a `state_dict` | Cell 16, deployable on 512 MB CPU | PASS |
| Medical disclaimer | first markdown cell | PASS |

**Next:** unzip the artifacts into `backend/artifacts/` and follow `DEPLOY.md`.

---

> **MEDICAL DISCLAIMER.** Educational project only. This model is **not a medical device**, has not been
> clinically validated, and must **never** be used for real diagnosis, triage, or treatment decisions.
> Always consult a qualified radiologist or physician.